# 🗜️ Context Compression Assistant

## What We're Building

A RAG pipeline that retrieves **many chunks** (high recall) and then
**compresses** them into a smaller, focused evidence set before sending
to the LLM.

## The Problem

```
Retrieve k=20 chunks  →  ~8000 tokens of context  →  LLM gets confused
Retrieve k=3 chunks   →  ~1200 tokens of context  →  Might miss relevant info
```

**Context compression** gives us the best of both worlds:
```
Retrieve k=20  →  Compress to top 4  →  ~1600 focused tokens  →  Better answers
```

## Three Compression Strategies

| Strategy | How | Speed | Quality |
|----------|-----|-------|--------|
| **LLM Reranker** | Score each chunk with LLM | Slow | Best |
| **Extractive** | Pull only relevant sentences | Medium | Good |
| **Abstractive** | LLM summarizes the chunks | Slow | Good for summaries |

## Stack
- **LangChain** — retrieval + compression
- **ChromaDB** — vector store
- **Ollama** — local LLM for reranking + answering

## Step 1 — Imports & Knowledge Base

In [ ]:
import re
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_community.vectorstores import Chroma
from langchain.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.schema import Document

print("All imports successful!")

In [ ]:
# Long document with many topics — retrieval will return many partially-relevant chunks
knowledge = """# Comprehensive Guide to Cloud Computing

## What is Cloud Computing?
Cloud computing delivers computing services over the internet: servers, storage,
databases, networking, software, analytics, and AI. Instead of owning physical
hardware, you rent resources from cloud providers on a pay-as-you-go basis.

## The Big Three Cloud Providers
AWS (Amazon Web Services) launched in 2006 and holds ~32% market share.
Key services: EC2 (compute), S3 (storage), RDS (databases), Lambda (serverless).
Azure (Microsoft) holds ~23% share. Strengths: enterprise integration, hybrid cloud.
GCP (Google Cloud) holds ~11% share. Strengths: data analytics, ML/AI, Kubernetes.

## Cloud Service Models
IaaS (Infrastructure): Raw compute, storage, networking. You manage the OS and up.
Examples: AWS EC2, Azure VMs, GCP Compute Engine.
PaaS (Platform): Managed runtime for apps. Provider manages OS, middleware.
Examples: Heroku, Google App Engine, Azure App Service.
SaaS (Software): Ready-to-use applications. Provider manages everything.
Examples: Gmail, Salesforce, Slack, Microsoft 365.

## Compute Services
Virtual Machines: Full OS instances. Good for legacy apps and full control.
Containers: Docker + Kubernetes. Lightweight, portable, fast startup.
Serverless: Functions triggered by events. Pay per execution. AWS Lambda, Azure Functions.
Bare Metal: Dedicated physical servers. For high-performance, compliance requirements.

## Storage Services
Object Storage: S3, GCS, Azure Blob. For files, images, backups. Highly durable.
Block Storage: EBS, Persistent Disks. Low-latency, attach to VMs.
File Storage: EFS, Cloud Filestore. Shared file systems, NFS compatible.
Archive Storage: S3 Glacier, Archive Storage. Cheap, slow retrieval (hours).

## Database Services
Relational: RDS, Cloud SQL, Azure SQL. PostgreSQL, MySQL, SQL Server managed.
NoSQL: DynamoDB, Firestore, Cosmos DB. Key-value, document, graph models.
Data Warehousing: Redshift, BigQuery, Synapse. Analytical workloads at petabyte scale.
In-Memory: ElastiCache (Redis), Memorystore. Microsecond latency caching.

## Networking
VPC (Virtual Private Cloud): Isolated network within the cloud. Define subnets,
route tables, security groups, and NAT gateways.
CDN: CloudFront, Cloud CDN, Azure CDN. Cache content at edge locations globally.
Load Balancing: ALB/NLB, Cloud Load Balancing. Distribute traffic across instances.
DNS: Route 53, Cloud DNS. Domain management and routing policies.

## Security Best Practices
Identity: Use IAM with least-privilege access. Enable MFA for all users.
Encryption: Encrypt at rest (AES-256) and in transit (TLS 1.3).
Network: Use VPCs, security groups, and WAF. Restrict public access.
Monitoring: CloudTrail, Cloud Audit Logs. Log all API calls.
Compliance: SOC 2, ISO 27001, HIPAA, GDPR. Use compliance-ready services.

## Cost Optimization
Right-sizing: Match instance size to actual workload needs.
Reserved Instances: 1-3 year commitments for 30-60% savings.
Spot/Preemptible: Use interruptible instances for fault-tolerant workloads. 60-90% savings.
Auto-scaling: Scale up during peaks, down during quiet periods.
Storage Tiers: Move cold data to archive storage for 80%+ savings.

## Serverless Computing
Benefits: No server management, auto-scaling, pay per request, fast deployment.
Limitations: Cold starts (100ms-3s), execution time limits (15 min on Lambda),
limited local storage, vendor lock-in.
Best for: Event processing, APIs, scheduled tasks, data transformation.
Not ideal for: Long-running processes, stateful applications, GPU workloads.

## Multi-Cloud Strategy
Benefits: Avoid vendor lock-in, leverage best-of-breed services.
Challenges: Increased complexity, different APIs, data transfer costs.
Tools: Terraform for IaC, Kubernetes for container portability.
Reality: Most companies have 80%+ on one provider, rest for specific use cases.
"""

# Create many small chunks (simulates a real corpus with lots of partial matches)
splitter = RecursiveCharacterTextSplitter(chunk_size=250, chunk_overlap=30, separators=["\n## ", "\n\n", "\n"])
docs = splitter.create_documents([knowledge])
embeddings = OllamaEmbeddings(model="nomic-embed-text-v2-moe")
vectorstore = Chroma.from_documents(docs, embeddings, collection_name="compression")

llm = ChatOllama(model="qwen3.5:9b", temperature=0.1)

print(f"Indexed {len(docs)} chunks (intentionally small for lots of partial matches)")

## Step 2 — Strategy 1: LLM-Based Reranking

Score each chunk's relevance to the query (0-10), then keep only the top ones.

In [ ]:
rerank_prompt = ChatPromptTemplate.from_template(
    """Rate how relevant this text chunk is to the query on a scale of 0-10.

Query: {query}

Chunk: {chunk}

Respond with ONLY a number from 0 to 10.

Relevance score:"""
)


def rerank_chunks(query: str, chunks: list[Document], top_k: int = 4) -> list[Document]:
    """Score each chunk with the LLM and return top-k."""
    chain = rerank_prompt | llm | StrOutputParser()
    scored = []

    for chunk in chunks:
        response = chain.invoke({"query": query, "chunk": chunk.page_content})
        # Extract number from response
        match = re.search(r'\b(10|[0-9])\b', response)
        score = int(match.group(1)) if match else 0
        scored.append((chunk, score))

    scored.sort(key=lambda x: x[1], reverse=True)
    return [chunk for chunk, _ in scored[:top_k]], scored


# Test reranking
query = "How can I reduce cloud costs?"
raw_chunks = vectorstore.similarity_search(query, k=10)

print(f"Query: {query}")
print(f"Retrieved {len(raw_chunks)} chunks, reranking...\n")

top_chunks, all_scored = rerank_chunks(query, raw_chunks, top_k=4)

print("All scores (before compression):")
for chunk, score in all_scored:
    preview = chunk.page_content[:60].replace('\n', ' ')
    marker = " ← KEPT" if chunk in top_chunks else ""
    print(f"  [{score:>2}/10] {preview}...{marker}")

## Step 3 — Strategy 2: Extractive Compression

Instead of keeping/dropping whole chunks, extract only the
**relevant sentences** from each chunk.

In [ ]:
extract_prompt = ChatPromptTemplate.from_template(
    """Given the query and the text chunk, extract ONLY the sentences that
are relevant to answering the query. Remove irrelevant sentences.

If no sentences are relevant, respond with "NONE".

Query: {query}

Chunk:
{chunk}

Relevant sentences only:"""
)


def extractive_compress(query: str, chunks: list[Document]) -> list[str]:
    """Extract only relevant sentences from each chunk."""
    chain = extract_prompt | llm | StrOutputParser()
    compressed = []

    for chunk in chunks:
        result = chain.invoke({"query": query, "chunk": chunk.page_content})
        if result.strip().upper() != "NONE" and len(result.strip()) > 10:
            compressed.append(result.strip())

    return compressed


# Test extractive compression
query = "What are the security best practices for cloud?"
raw_chunks = vectorstore.similarity_search(query, k=8)

print(f"Query: {query}")
print(f"Retrieved {len(raw_chunks)} chunks ({sum(len(c.page_content) for c in raw_chunks)} chars)\n")

compressed = extractive_compress(query, raw_chunks)

print(f"After extraction: {len(compressed)} relevant segments ({sum(len(c) for c in compressed)} chars)")
print(f"Compression ratio: {sum(len(c) for c in compressed) / sum(len(c.page_content) for c in raw_chunks):.0%}\n")
for i, segment in enumerate(compressed, 1):
    print(f"  [{i}] {segment[:80]}...")

## Step 4 — Strategy 3: Abstractive Summary

Summarize all retrieved chunks into a concise evidence paragraph.

In [ ]:
summarize_prompt = ChatPromptTemplate.from_template(
    """Summarize the following chunks into a concise evidence paragraph
that contains ONLY information relevant to the query.

Keep specific facts, numbers, and names. Remove redundancy.
Target length: 3-5 sentences.

Query: {query}

Chunks:
{chunks}

Focused summary:"""
)


def abstractive_compress(query: str, chunks: list[Document]) -> str:
    """Summarize chunks into a focused evidence paragraph."""
    chunk_text = "\n\n---\n\n".join(c.page_content for c in chunks)
    chain = summarize_prompt | llm | StrOutputParser()
    return chain.invoke({"query": query, "chunks": chunk_text})


# Test
query = "Compare compute options in the cloud"
raw_chunks = vectorstore.similarity_search(query, k=10)

summary = abstractive_compress(query, raw_chunks)
print(f"Query: {query}")
print(f"Input: {len(raw_chunks)} chunks ({sum(len(c.page_content) for c in raw_chunks)} chars)")
print(f"Output: {len(summary)} chars\n")
print(summary)

## Step 5 — Compare All Strategies End-to-End

In [ ]:
answer_prompt = ChatPromptTemplate.from_template(
    """Answer the question using ONLY the provided context.

Context:
{context}

Question: {question}

Answer:"""
)
answer_chain = answer_prompt | llm | StrOutputParser()


def compare_strategies(question: str) -> None:
    """Run question through all 4 approaches and compare."""
    print(f"❓ {question}")
    print("=" * 60)

    # Retrieve many chunks (same for all)
    raw_chunks = vectorstore.similarity_search(question, k=10)
    raw_context = "\n\n".join(c.page_content for c in raw_chunks)
    print(f"\nRetrieved {len(raw_chunks)} chunks ({len(raw_context)} chars)\n")

    # 1. No compression (baseline)
    print("📌 Strategy 1: No Compression (all 10 chunks)")
    ans1 = answer_chain.invoke({"context": raw_context, "question": question})
    print(f"  Context size: {len(raw_context)} chars")
    print(f"  Answer: {ans1[:150]}...\n")

    # 2. LLM Reranking (top 4)
    print("📌 Strategy 2: LLM Reranking (top 4 of 10)")
    reranked, _ = rerank_chunks(question, raw_chunks, top_k=4)
    ctx2 = "\n\n".join(c.page_content for c in reranked)
    ans2 = answer_chain.invoke({"context": ctx2, "question": question})
    print(f"  Context size: {len(ctx2)} chars ({len(ctx2)/len(raw_context):.0%} of original)")
    print(f"  Answer: {ans2[:150]}...\n")

    # 3. Extractive
    print("📌 Strategy 3: Extractive Compression")
    extracted = extractive_compress(question, raw_chunks)
    ctx3 = "\n\n".join(extracted)
    ans3 = answer_chain.invoke({"context": ctx3, "question": question})
    print(f"  Context size: {len(ctx3)} chars ({len(ctx3)/len(raw_context):.0%} of original)")
    print(f"  Answer: {ans3[:150]}...\n")

    # 4. Abstractive
    print("📌 Strategy 4: Abstractive Summary")
    ctx4 = abstractive_compress(question, raw_chunks)
    ans4 = answer_chain.invoke({"context": ctx4, "question": question})
    print(f"  Context size: {len(ctx4)} chars ({len(ctx4)/len(raw_context):.0%} of original)")
    print(f"  Answer: {ans4[:150]}...")

    print("\n" + "=" * 60)

In [ ]:
compare_strategies("What are the main differences between serverless and containers?")

In [ ]:
compare_strategies("How can I optimize cloud costs? Give specific strategies.")

## 🧠 Key Concepts Recap

| Strategy | Context Reduction | Preserves | Best For |
|----------|------------------|-----------|----------|
| **No compression** | 0% | Everything | Short contexts |
| **LLM Reranking** | ~60% | Best whole chunks | Precision-focused |
| **Extractive** | ~70% | Key sentences | Fact-heavy questions |
| **Abstractive** | ~80%+ | Core meaning | Summary questions |

## 🔧 Customization Ideas

- **Cross-encoder reranker**: Use a fine-tuned BERT reranker (faster than LLM)
- **Cohere Rerank API**: Production-grade reranking as a service
- **Token budget**: Set a max token budget and compress to fit
- **Pipeline combo**: Rerank first, then extract sentences from top chunks